# User Repeat Transaction Prediction

Business goal: predict which users are likely to repeat exchange activity in the next 30 days.

This model can support retention campaigns, lifecycle nudges, and product feature prompts for an FX fintech platform.

## 1. Setup

In [ ]:
from pathlib import Path
import sqlite3
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from xgboost import XGBClassifier

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "fx_fintech_product_analytics.db"
OUTPUT_CSV_DIR = PROJECT_ROOT / "outputs" / "csv"
MODEL_DIR = PROJECT_ROOT / "models"

OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH

## 2. Inspect Modeling Table

In [ ]:
conn = sqlite3.connect(DB_PATH)

columns = pd.read_sql_query("""
    PRAGMA table_info(modeling_user_month_snapshots);
""", conn)

columns

In [ ]:
summary = pd.read_sql_query("""
    SELECT
        COUNT(*) AS snapshot_rows,
        COUNT(DISTINCT user_id) AS users,
        MIN(observation_date) AS min_observation_date,
        MAX(observation_date) AS max_observation_date,
        SUM(target_repeat_30d) AS positive_targets,
        ROUND(
            100.0 * SUM(target_repeat_30d) / COUNT(*),
            2
        ) AS positive_target_rate,
        ROUND(AVG(recency_days), 2) AS avg_recency_days,
        ROUND(AVG(transactions_90d), 2) AS avg_transactions_90d,
        ROUND(AVG(failed_ratio_90d), 4) AS avg_failed_ratio_90d
    FROM modeling_user_month_snapshots;
""", conn)

summary

In [ ]:
sample = pd.read_sql_query("""
    SELECT *
    FROM modeling_user_month_snapshots
    LIMIT 10;
""", conn)

sample

In [ ]:
df = pd.read_sql_query("""
    SELECT *
    FROM modeling_user_month_snapshots;
""", conn)

conn.close()

df.head()

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing_summary["missing_rate"] = missing_summary["missing_count"] / len(df)

missing_summary.sort_values("missing_count", ascending=False)

In [ ]:
target_balance = (
    df["target_repeat_30d"]
    .value_counts(normalize=False)
    .rename_axis("target_repeat_30d")
    .reset_index(name="rows")
)

target_balance["share"] = target_balance["rows"] / target_balance["rows"].sum()

target_balance

In [ ]:
df["observation_date"] = pd.to_datetime(df["observation_date"])

monthly_rows = (
    df.groupby(df["observation_date"].dt.to_period("M"))
    .agg(
        rows=("user_id", "count"),
        users=("user_id", "nunique"),
        positive_rate=("target_repeat_30d", "mean"),
    )
    .reset_index()
)

monthly_rows["observation_month"] = monthly_rows["observation_date"].astype(str)
monthly_rows = monthly_rows.drop(columns="observation_date")

monthly_rows.head(), monthly_rows.tail()

## 3. Feature Selection and Time-Based Split

In [ ]:
df.columns.tolist()

In [ ]:
target_col = "target_repeat_30d"

feature_cols = [
    "recency_days",
    "transactions_90d",
    "transactions_all",
    "total_volume_90d",
    "failed_ratio_90d",
]

feature_cols

In [ ]:
df[feature_cols].dtypes.sort_values()

In [ ]:
df = df.sort_values("observation_date").copy()

cutoff_date = pd.Timestamp("2025-10-31")

train_df = df[df["observation_date"] <= cutoff_date].copy()
test_df = df[df["observation_date"] > cutoff_date].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Cutoff date:", cutoff_date)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

In [ ]:
missing_features = (
    X_train.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_count"})
)

missing_features["missing_rate"] = missing_features["missing_count"] / len(X_train)

missing_features.sort_values("missing_count", ascending=False)

## 4. Baseline and Candidate Models

In [ ]:
baseline_pred = np.zeros(len(y_test))

baseline_metrics = {
    "model": "Majority class baseline",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0),
    "roc_auc": 0.5,
}

baseline_metrics

In [ ]:
logreg_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

logreg_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_test)
logreg_proba = logreg_model.predict_proba(X_test)[:, 1]

logreg_metrics = {
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, logreg_pred),
    "precision": precision_score(y_test, logreg_pred),
    "recall": recall_score(y_test, logreg_pred),
    "f1": f1_score(y_test, logreg_pred),
    "roc_auc": roc_auc_score(y_test, logreg_proba),
}

logreg_metrics

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "model": "Random Forest",
    "accuracy": accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred),
    "recall": recall_score(y_test, rf_pred),
    "f1": f1_score(y_test, rf_pred),
    "roc_auc": roc_auc_score(y_test, rf_proba),
}

rf_metrics

In [ ]:
gb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    random_state=42,
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)[:, 1]

gb_metrics = {
    "model": "Gradient Boosting",
    "accuracy": accuracy_score(y_test, gb_pred),
    "precision": precision_score(y_test, gb_pred),
    "recall": recall_score(y_test, gb_pred),
    "f1": f1_score(y_test, gb_pred),
    "roc_auc": roc_auc_score(y_test, gb_proba),
}

gb_metrics

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_metrics = {
    "model": "XGBoost",
    "accuracy": accuracy_score(y_test, xgb_pred),
    "precision": precision_score(y_test, xgb_pred),
    "recall": recall_score(y_test, xgb_pred),
    "f1": f1_score(y_test, xgb_pred),
    "roc_auc": roc_auc_score(y_test, xgb_proba),
}

xgb_metrics

In [ ]:
model_metrics = pd.DataFrame([
    baseline_metrics,
    logreg_metrics,
    rf_metrics,
    gb_metrics,
    xgb_metrics,
])

model_metrics.sort_values("f1", ascending=False)

## 5. Final Model Evaluation: XGBoost

XGBoost is selected as the final model because it provides the strongest F1 score while maintaining high recall. This makes it useful for identifying users likely to repeat exchange activity in the next 30 days.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_pred,
    display_labels=["No repeat", "Repeat"],
    cmap="Blues",
    ax=ax,
)

ax.set_title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, xgb_pred),
    index=["Actual no repeat", "Actual repeat"],
    columns=["Predicted no repeat", "Predicted repeat"],
)

confusion_matrix_df

In [ ]:
report = classification_report(
    y_test,
    xgb_pred,
    target_names=["No repeat", "Repeat"],
    output_dict=True,
)

classification_report_df = pd.DataFrame(report).T

classification_report_df

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.barh(
    feature_importance.sort_values("importance")["feature"],
    feature_importance.sort_values("importance")["importance"],
    color="#2F6F73",
)

ax.set_title("XGBoost Feature Importance")
ax.set_xlabel("Importance")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Export Model Outputs

In [ ]:
prediction_output = test_df[["user_id", "observation_date", target_col]].copy()
prediction_output["repeat_probability"] = xgb_proba
prediction_output["predicted_repeat_30d"] = xgb_pred

prediction_output.head()

In [ ]:
model_metrics.to_csv(OUTPUT_CSV_DIR / "user_repeat_model_metrics.csv", index=False)
feature_importance.to_csv(OUTPUT_CSV_DIR / "user_repeat_feature_importance.csv", index=False)
prediction_output.to_csv(OUTPUT_CSV_DIR / "user_repeat_predictions.csv", index=False)
classification_report_df.to_csv(OUTPUT_CSV_DIR / "user_repeat_classification_report.csv")
confusion_matrix_df.to_csv(OUTPUT_CSV_DIR / "user_repeat_confusion_matrix.csv")

with open(MODEL_DIR / "user_repeat_xgboost_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)

print("Exported model outputs to:", OUTPUT_CSV_DIR)
print("Saved model to:", MODEL_DIR / "user_repeat_xgboost_model.pkl")

## 7. Business Interpretation

The repeat prediction model identifies users who are more likely to repeat exchange activity in the next 30 days.

Product and lifecycle teams can use this output to:

- prioritize retention campaigns,
- trigger feature education prompts,
- identify users who may benefit from target-rate or rate-alert nudges,
- segment users by repeat probability for campaign testing.

This model should be treated as a product operations signal, not as financial advice.